# Customer Churn Prediction — Portfolio Version

**Project goal:** analyze telecom customer churn, train and compare classification models, evaluate the errors that matter to the business, and identify customer characteristics associated with churn.

**Dataset:** IBM Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`).

This version is designed to be strong enough for a portfolio while keeping each step explainable. It uses Pandas, Matplotlib, Seaborn, and Scikit-learn, plus Joblib only for saving the final model.

## Workflow
1. Load and inspect the data
2. Explore churn patterns
3. Clean the data
4. Split into training and test sets
5. Preprocess features correctly
6. Train and compare three models
7. Evaluate the best model
8. Inspect feature importance
9. Save the model and test predictions

**Before running:** put the CSV file in the same folder as this notebook, or change the path in Step 1. Run cells from top to bottom.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import joblib

sns.set_theme(style="whitegrid")

## 1. Load the dataset

A DataFrame is a table: rows are customers and columns are their information.

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Dataset shape:", df.shape)
display(df.head())

## 2. Inspect the data

Check column types, missing values, duplicate rows, and the target distribution before training anything.

In [ ]:
print("Column types:")
display(df.dtypes.to_frame("data_type"))

print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

print("Duplicate rows:", df.duplicated().sum())
print("\nChurn counts:")
display(df["Churn"].value_counts())
print("\nChurn proportions:")
display((df["Churn"].value_counts(normalize=True) * 100).round(1))

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn", hue="Churn", legend=False)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

## 3. Explore possible churn patterns

These charts are descriptive, not proof that a feature causes churn. We look at contract type, internet service, payment method, tenure, and monthly charges.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

for ax, col in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    churn_rate = df.groupby(col, observed=False)["Churn"].apply(
        lambda values: (values == "Yes").mean() * 100
    ).sort_values()
    churn_rate.plot(kind="barh", ax=ax)
    ax.set_title(f"Churn rate by {col}")
    ax.set_xlabel("Customers who churned (%)")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
df_plot = df.copy()
df_plot["TotalCharges"] = pd.to_numeric(df_plot["TotalCharges"], errors="coerce")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.histplot(data=df_plot, x=col, hue="Churn", bins=30, element="step",
                 stat="density", common_norm=False, ax=ax)
    ax.set_title(f"{col} by churn")
plt.tight_layout()
plt.show()

## 4. Clean and prepare the target

- `customerID` is an identifier, so we exclude it from the predictors.
- `TotalCharges` is stored as text in this dataset; convert it to a numeric column.
- Blank values become missing values. We will fill missing numeric/categorical values inside the preprocessing pipeline, using training data only.
- The target `Churn` becomes `1` for Yes and `0` for No.

In [ ]:
data = df.drop(columns=["customerID"]).copy()

data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

print("Target values:")
print(data["Churn"].value_counts())
print("\nRows:", len(data))
print("Missing TotalCharges:", data["TotalCharges"].isna().sum())

## 5. Separate features and target, then split

- `X` = information used to predict churn.
- `y` = the answer the model learns to predict.
- The test set is kept aside until evaluation.
- `stratify=y` keeps a similar proportion of churners in training and testing.

Preprocessing will be fitted only on the training data. This helps prevent **data leakage** from the test set into training.

In [ ]:
X = data.drop(columns=["Churn"])
y = data["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training customers:", len(X_train))
print("Testing customers:", len(X_test))
print("Churn rate in training set:", round(y_train.mean() * 100, 1), "%")
print("Churn rate in test set:", round(y_test.mean() * 100, 1), "%")

## 6. Preprocess numeric and categorical columns

Machine learning models need numeric inputs.

- Numeric columns: fill missing values with the median and standardize their scales.
- Categorical columns: fill missing values with the most common category, then one-hot encode them into 0/1 columns.

A `Pipeline` keeps preprocessing and model training together. Each model learns preprocessing from the training data only, then applies the same transformations to test data.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessing, numeric_features),
    ("categorical", categorical_preprocessing, categorical_features)
])

print("Numeric features:", numeric_features)
print("Number of categorical columns:", len(categorical_features))

## 7. Train and compare three models

We compare different approaches:
- **Logistic Regression:** a useful baseline and relatively easy to interpret.
- **Random Forest:** combines many decision trees.
- **Gradient Boosting:** builds trees sequentially, with later trees trying to correct earlier errors.

We compare accuracy, precision, recall, F1-score, and ROC-AUC. Because churners are the minority class, accuracy alone is not enough. Recall tells us what fraction of actual churners were detected.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=3,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=2, random_state=42
    )
}

trained_models = {}
results = []
test_probabilities = {}

for name, classifier in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", classifier)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    trained_models[name] = pipeline
    test_probabilities[name] = probabilities

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(results_df.round(3))

## 8. Compare model performance visually

These charts make it easier to compare models. The ROC curve shows the trade-off between detecting churners and incorrectly flagging customers who would stay. A curve closer to the top-left is generally better; ROC-AUC summarizes this ranking performance.

In [ ]:
metrics_to_plot = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]]
metrics_to_plot.plot(kind="bar", figsize=(12, 5), ylim=(0, 1))
plt.title("Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, probabilities in test_probabilities.items():
    RocCurveDisplay.from_predictions(y_test, probabilities, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random baseline")
ax.set_title("ROC Curves — Model Comparison")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Evaluate the best model

For this notebook, we select the model with the highest ROC-AUC on the held-out test set for demonstration. In a more rigorous model-selection workflow, choose the model using cross-validation on the training data, then evaluate the selected model on the test set only once.

The threshold used by `predict()` is the model's default. A real retention team may choose a different threshold depending on the cost of missed churners versus unnecessary retention offers.

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
best_predictions = best_model.predict(X_test)
best_probabilities = best_model.predict_proba(X_test)[:, 1]

print("Selected model:", best_name)
print("\nClassification report:")
print(classification_report(
    y_test, best_predictions,
    target_names=["Stayed", "Churned"],
    zero_division=0
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions,
    display_labels=["Stayed", "Churned"],
    cmap="Blues"
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, best_predictions).ravel()
print("True negatives (correctly predicted stayed):", tn)
print("False positives (flagged but stayed):", fp)
print("False negatives (missed churners):", fn)
print("True positives (correctly detected churners):", tp)

## 10. Understand important features

For tree-based models, feature importance estimates how much each encoded feature contributes to reducing errors across the trees. It can help guide further investigation, but it does **not** prove causation and is not the same as a business explanation.

This cell works when the selected model is Random Forest or Gradient Boosting.

In [ ]:
if best_name in ["Random Forest", "Gradient Boosting"]:
    fitted_preprocessor = best_model.named_steps["preprocessing"]
    fitted_classifier = best_model.named_steps["model"]

    feature_names = fitted_preprocessor.get_feature_names_out()
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": fitted_classifier.feature_importances_
    }).sort_values("Importance", ascending=False).head(15)

    display(importance_df)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, x="Importance", y="Feature")
    plt.title(f"Top 15 Feature Importances — {best_name}")
    plt.tight_layout()
    plt.show()
else:
    print("The selected Logistic Regression model does not provide tree feature_importances_.")
    print("You can inspect its coefficients in a later extension.")

## 11. Export predictions and save the trained pipeline

`joblib` saves a Python object to a file so it can be loaded later without training again. We save the **entire pipeline**, including preprocessing and the model, so future data receives the same transformations.

The output files are written to the current notebook folder.

In [ ]:
prediction_output = X_test.copy()
prediction_output["ActualChurn"] = y_test
prediction_output["ChurnProbability"] = best_probabilities
prediction_output["PredictedChurn"] = best_predictions
prediction_output["RiskSegment"] = pd.cut(
    best_probabilities,
    bins=[-0.001, 0.30, 0.60, 1.001],
    labels=["Low", "Medium", "High"]
)

prediction_output.to_csv("churn_test_predictions.csv", index=False)
joblib.dump(best_model, "churn_prediction_pipeline.joblib")

print("Saved: churn_test_predictions.csv")
print("Saved: churn_prediction_pipeline.joblib")
print("Best model selected:", best_name)

## 12. Example: predict one customer's churn risk

Use one customer from the test set as an example. The probability is the model's estimated churn probability, not a guarantee of what will happen.

In [ ]:
one_customer = X_test.iloc[[0]]
churn_probability = best_model.predict_proba(one_customer)[0, 1]
churn_prediction = best_model.predict(one_customer)[0]

print("Estimated churn probability:", f"{churn_probability:.1%}")
print("Predicted outcome:", "Churn" if churn_prediction == 1 else "No churn")
print("Actual outcome:", "Churn" if y_test.iloc[0] == 1 else "No churn")

## Final summary

This project demonstrates:
- Exploratory data analysis and visual storytelling
- Data cleaning and categorical encoding
- A train/test split that preserves the target balance
- A preprocessing pipeline to reduce data leakage
- Comparison of three classification algorithms
- Evaluation using precision, recall, F1-score, ROC-AUC, and a confusion matrix
- Feature-importance analysis
- Exporting predictions and saving a reusable model pipeline

### Be ready to explain these in an interview
1. Why is `Churn` the target, and what goes into `X`?
2. Why do we split into training and testing data?
3. Why do categorical values need encoding?
4. Why can accuracy be misleading when classes are imbalanced?
5. What is the difference between precision and recall?
6. Why do we fit preprocessing only on the training data?
7. What does ROC-AUC measure?
8. What does saving a pipeline with Joblib allow us to do?

**CV honesty:** describe it as a churn prediction and model-comparison project. Only claim deployment, API integration, or production monitoring if you actually implement those separately.